## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [59]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [60]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kabkot"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [61]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [62]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Diskrepansi Prov vs KabKot")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'provinsi', 'adhb_pdrb_q1-25', 'adhb_pdrb_q2-25', 'adhb_pdrb_q3-25', 'adhb_pkrt_q1-25', 'adhb_pkrt_q2-25', 'adhb_pkrt_q3-25', 'adhb_pklnprt_q1-25', 'adhb_pklnprt_q2-25', 'adhb_pklnprt_q3-25', 'adhb_pkp_q1-25', 'adhb_pkp_q2-25', 'adhb_pkp_q3-25', 'adhb_pmtb_q1-25', 'adhb_pmtb_q2-25', 'adhb_pmtb_q3-25', 'adhb_pi_q1-25', 'adhb_pi_q2-25', 'adhb_pi_q3-25', 'adhb_net_ekspor_q1-25', 'adhb_net_ekspor_q2-25', 'adhb_net_ekspor_q3-25', 'adhk_pdrb_q1-25', 'adhk_pdrb_q2-25', 'adhk_pdrb_q3-25', 'adhk_pkrt_q1-25', 'adhk_pkrt_q2-25', 'adhk_pkrt_q3-25', 'adhk_pklnprt_q1-25', 'adhk_pklnprt_q2-25', 'adhk_pklnprt_q3-25', 'adhk_pkp_q1-25', 'adhk_pkp_q2-25', 'adhk_pkp_q3-25', 'adhk_pmtb_q1-25', 'adhk_pmtb_q2-25', 'adhk_pmtb_q3-25', 'adhk_pi_q1-25', 'adhk_pi_q2-25', 'adhk_pi_q3-25', 'adhk_net_ekspor_q1-25', 'adhk_net_ekspor_q2-25', 'adhk_net_ekspor_q3-25', 'distribusi_pdrb_q1-25', 'distribusi_pdrb_q1-25.1', 'distribusi_pdrb_q2-25', 'distribusi_pdrb_q2-25.1', 'distribusi_pdrb_q3-25', 'distribusi_pdrb

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [63]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2025
            if any(q in col for q in ["q1", "q2"]):
                if pd.notna(val) and (val < -5 or val > 5):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # ADHB dan ADHK Q3 2025
            elif "q3" in col:
                if pd.notna(val) and (val < -3 or val > 3):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1, Q2, dan Q3 2025
            if pd.notna(val) and (val < -5 or val > 5):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                elif "adhk" in col:
                    evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,  # tambahan biar tau sumbernya
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

     kode           provinsi periode      nilai                  kolom  \
0      92   Papua Barat Daya   q2-25   5.257010     adhb_pklnprt_q2-25   
1      96       Papua Tengah   q3-25  -5.207338     adhb_pklnprt_q3-25   
2      33        Jawa Tengah   q3-25  -7.050902         adhb_pkp_q3-25   
3      82       Maluku Utara   q3-25  16.483375         adhb_pkp_q3-25   
4      95      Papua Selatan   q3-25   5.228286         adhb_pkp_q3-25   
..    ...                ...     ...        ...                    ...   
162    73   Sulawesi Selatan   q3-25 -36.465970  adhk_net_ekspor_q3-25   
163    74  Sulawesi Tenggara   q3-25  -5.825411  adhk_net_ekspor_q3-25   
164    75          Gorontalo   q3-25   6.816240  adhk_net_ekspor_q3-25   
165    82       Maluku Utara   q3-25  -6.639632  adhk_net_ekspor_q3-25   
166    95      Papua Selatan   q3-25   7.023643  adhk_net_ekspor_q3-25   

                                              evaluasi  
0    Diskrepansi ADHB antara Provinsi dan total Kab...

#### 3. Evaluasi distribusi

In [64]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1,Q2,Q3 2025
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if selisih > 5:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-5 point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

   kode       provinsi periode      nilai  \
0    95  Papua Selatan   q3-25 -31.325496   

                                               kolom  \
0  distribusi_net_ekspor_q3-25 vs distribusi_net_...   

                                            evaluasi  
0  Diskrepansi Distribusi antara Provinsi dan tot...  


#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [65]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2025
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            print(selisih)
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    elif "q3" in col0:
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.01:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                            }
                            evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2025
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 1:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    elif "q3" in col0:
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,05 point)"
                            }
                            evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,3 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

0.6882876094443962
0.7796058789315641
0.770731569062451
1.3214357951628006
0.8566447575813765
     kode             provinsi periode     nilai  \
0      11                 Aceh   q1-25 -5.031793   
1      15                Jambi   q1-25 -2.569168   
2      16     Sumatera Selatan   q1-25 -0.599034   
3      53  Nusa Tenggara Timur   q1-25 -5.802210   
4      63   Kalimantan Selatan   q1-25 -4.777180   
..    ...                  ...     ...       ...   
707    64     Kalimantan Timur   q3-25  6.506066   
708    65     Kalimantan Utara   q3-25  2.430408   
709    91          Papua Barat   q3-25 -0.014948   
710    92     Papua Barat Daya   q3-25 -4.318997   
711    97     Papua Pegunungan   q3-25  2.215418   

                                                 kolom  \
0             q-to-q_pdrb_q1-25 vs q-to-q_pdrb_q1-25.1   
1             q-to-q_pdrb_q1-25 vs q-to-q_pdrb_q1-25.1   
2             q-to-q_pdrb_q1-25 vs q-to-q_pdrb_q1-25.1   
3             q-to-q_pdrb_q1-25 vs q-to-q_pdrb_q1

## Menggabungkan seluruh hasil evaluasi

In [66]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

   kode          provinsi periode      nilai               kolom  \
0    92  Papua Barat Daya   q2-25   5.257010  adhb_pklnprt_q2-25   
1    96      Papua Tengah   q3-25  -5.207338  adhb_pklnprt_q3-25   
2    33       Jawa Tengah   q3-25  -7.050902      adhb_pkp_q3-25   
3    82      Maluku Utara   q3-25  16.483375      adhb_pkp_q3-25   
4    95     Papua Selatan   q3-25   5.228286      adhb_pkp_q3-25   

                                            evaluasi  
0  Diskrepansi ADHB antara Provinsi dan total Kab...  
1  Diskrepansi ADHB antara Provinsi dan total Kab...  
2  Diskrepansi ADHB antara Provinsi dan total Kab...  
3  Diskrepansi ADHB antara Provinsi dan total Kab...  
4  Diskrepansi ADHB antara Provinsi dan total Kab...  


## Menuliskan hasil evaluasi kedalam file excel

In [67]:
write_to_excel(evaluasi, "provinsi")

File baru dibuat: data/output/evaluasi_92.xlsx, dengan sheet 9200
File baru dibuat: data/output/evaluasi_96.xlsx, dengan sheet 9600
File baru dibuat: data/output/evaluasi_33.xlsx, dengan sheet 3300
File baru dibuat: data/output/evaluasi_82.xlsx, dengan sheet 8200
File baru dibuat: data/output/evaluasi_95.xlsx, dengan sheet 9500
File baru dibuat: data/output/evaluasi_62.xlsx, dengan sheet 6200
File baru dibuat: data/output/evaluasi_11.xlsx, dengan sheet 1100
File baru dibuat: data/output/evaluasi_15.xlsx, dengan sheet 1500
File baru dibuat: data/output/evaluasi_34.xlsx, dengan sheet 3400
File baru dibuat: data/output/evaluasi_61.xlsx, dengan sheet 6100
File baru dibuat: data/output/evaluasi_73.xlsx, dengan sheet 7300
File baru dibuat: data/output/evaluasi_18.xlsx, dengan sheet 1800
File baru dibuat: data/output/evaluasi_16.xlsx, dengan sheet 1600
File baru dibuat: data/output/evaluasi_32.xlsx, dengan sheet 3200
File baru dibuat: data/output/evaluasi_64.xlsx, dengan sheet 6400
File baru 

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [68]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Revisi & Growth")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-25_ril', 'adhb_pdrb_q1-25_rev', 'adhb_pdrb_q2-25_ril', 'adhb_pdrb_q2-25_rev', 'adhb_pdrb_q3-25', 'adhb_pkrt_q1-25_ril', 'adhb_pkrt_q1-25_rev', 'adhb_pkrt_q2-25_ril', 'adhb_pkrt_q2-25_rev', 'adhb_pkrt_q3-25', 'adhb_pklnprt_q1-25_ril', 'adhb_pklnprt_q1-25_rev', 'adhb_pklnprt_q2-25_ril', 'adhb_pklnprt_q2-25_rev', 'adhb_pklnprt_q3-25', 'adhb_pkp_q1-25_ril', 'adhb_pkp_q1-25_rev', 'adhb_pkp_q2-25_ril', 'adhb_pkp_q2-25_rev', 'adhb_pkp_q3-25', 'adhb_pmtb_q1-25_ril', 'adhb_pmtb_q1-25_rev', 'adhb_pmtb_q2-25_ril', 'adhb_pmtb_q2-25_rev', 'adhb_pmtb_q3-25', 'adhb_pi_q1-25_ril', 'adhb_pi_q1-25_rev', 'adhb_pi_q2-25_ril', 'adhb_pi_q2-25_rev', 'adhb_pi_q3-25', 'adhb_net_ekspor_q1-25_ril', 'adhb_net_ekspor_q1-25_rev', 'adhb_net_ekspor_q2-25_ril', 'adhb_net_ekspor_q2-25_rev', 'adhb_net_ekspor_q3-25', 'adhk_pdrb_q1-25_ril', 'adhk_pdrb_q1-25_rev', 'adhk_pdrb_q2-25_ril', 'adhk_pdrb_q2-25_rev', 'adhk_pdrb_q3-25', 'adhk_pkrt_q1-25_ril', 'adhk_pkrt_q1-25_rev', 'adhk_pkr

#### 2. Evaluasi revisi ADHB dan ADHK

In [69]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.3:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-30%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

    kode             kabupaten/kota periode         nilai  \
0   1803  Kabupaten Lampung Selatan   q1-25  -2121.116913   
1   7211     Kabupaten Banggai Laut   q1-25  45846.310000   
2   7271                  Kota Palu   q1-25  22260.560000   
3   7372              Kota Parepare   q1-25 -20908.313313   
4   7601           Kabupaten Majene   q1-25 -56427.640000   
..   ...                        ...     ...           ...   
61  7372              Kota Parepare   q2-25 -48209.114341   
62  7401            Kabupaten Buton   q2-25 -13672.084071   
63  7601           Kabupaten Majene   q2-25  -1770.177525   
64  8203   Kabupaten Kepulauan Sula   q2-25 -11257.566156   
65  9701            Kabupaten Nduga   q2-25   1246.853210   

                                                kolom  \
0              adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
1              adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
2              adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
3              adhb_pi_q1-25_ril vs adh

#### 3. Evaluasi revisi distribusi

In [70]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 5:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-5 point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

    kode           kabupaten/kota periode      nilai  \
0   8105  Kabupaten Kepulauan Aru   q1-25   0.000000   
1   8105  Kabupaten Kepulauan Aru   q2-25   0.000000   
2   9102        Kabupaten Kaimana   q1-25  41.028173   
3   6111   Kabupaten Kayong Utara   q2-25  91.757249   
4   3403    Kabupaten Gunungkidul   q1-25  45.909973   
5   9102        Kabupaten Kaimana   q1-25  -1.474612   
6   9701          Kabupaten Nduga   q1-25   0.431125   
7   3403    Kabupaten Gunungkidul   q2-25  47.253113   
8   6111   Kabupaten Kayong Utara   q2-25 -65.020222   
9   7317           Kabupaten Luwu   q2-25  -1.035883   
10  9701          Kabupaten Nduga   q2-25   0.435696   

                                                kolom  \
0   distribusi_pkrt_q1-25_ril vs distribusi_pkrt_q...   
1   distribusi_pkrt_q2-25_ril vs distribusi_pkrt_q...   
2   distribusi_pkp_q1-25_ril vs distribusi_pkp_q1-...   
3   distribusi_pmtb_q2-25_ril vs distribusi_pmtb_q...   
4   distribusi_net_ekspor_q1-25_ril vs dis

#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [71]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-3 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-4 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        if v1 < -2 or v1 > 2:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "SoG PI lebih dari +-2 persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

     kode                kabupaten/kota periode       nilai  \
0    1606      Kabupaten Musi Banyuasin   q1-25   -6.605874   
1    1606      Kabupaten Musi Banyuasin   q2-25   -3.452782   
2    9701               Kabupaten Nduga   q2-25   -1.243570   
3    6471               Kota Balikpapan   q1-25  -68.642162   
4    6471               Kota Balikpapan   q1-25  423.981952   
..    ...                           ...     ...         ...   
511  8106  Kabupaten Seram Bagian Barat   q1-25    0.263607   
512  9701               Kabupaten Nduga   q1-25   59.779366   
513  5206                Kabupaten Bima   q2-25   -0.083687   
514  7317                Kabupaten Luwu   q2-25   -0.070716   
515  7372                 Kota Parepare   q2-25    0.724427   

                                                 kolom  \
0       q-to-q_pdrb_q1-25_ril vs q-to-q_pdrb_q1-25_rev   
1       q-to-q_pdrb_q2-25_ril vs q-to-q_pdrb_q2-25_rev   
2       q-to-q_pdrb_q2-25_ril vs q-to-q_pdrb_q2-25_rev   
3       q-t

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [72]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kabkot" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kabkot dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kabkot")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            if pd.notna(val) and (val < -2 or val > 2):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,
                    "evaluasi": "SoG PI lebih dari +-2 persen"
                }
                evaluasi_value_growth.append(record)

# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

sog_q_pi_q3-25 3 -2.214855468738683
sog_q_pi_q3-25 5 2.298097316203788
sog_q_pi_q3-25 21 2.690011607195602
sog_q_pi_q3-25 85 -4.584899201275705
sog_q_pi_q3-25 115 -2.494050789639586
sog_q_pi_q3-25 116 -2.3369044429139363
sog_q_pi_q3-25 117 -2.60461225146946
sog_q_pi_q3-25 118 -3.273919384681278
sog_q_pi_q3-25 119 -3.047683742182399
sog_q_pi_q3-25 120 -4.478263128729859
sog_q_pi_q3-25 121 -2.258830634757059
sog_q_pi_q3-25 122 -2.637425593027385
sog_q_pi_q3-25 123 -2.228960639717166
sog_q_pi_q3-25 126 -2.515274641535608
sog_q_pi_q3-25 129 -2.255476239575618
sog_q_pi_q3-25 147 -3.424123927219386
sog_q_pi_q3-25 148 -4.4371505565073885
sog_q_pi_q3-25 153 -2.162431979623555
sog_q_pi_q3-25 220 -2.326938370535237
sog_q_pi_q3-25 223 -3.8724069857435213
sog_q_pi_q3-25 271 -2.8891726088350334
sog_q_pi_q3-25 288 4.962353544424413
sog_q_pi_q3-25 296 -2.0422822888703136
sog_q_pi_q3-25 297 -2.087246408832149
sog_q_pi_q3-25 298 -2.3816423464078307
sog_q_pi_q3-25 304 -2.574537011518693
sog_q_pi_q3-25 3

## Menggabungkan seluruh hasil evaluasi

In [73]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

   kode             kabupaten/kota periode         nilai  \
0  1803  Kabupaten Lampung Selatan   q1-25  -2121.116913   
1  7211     Kabupaten Banggai Laut   q1-25  45846.310000   
2  7271                  Kota Palu   q1-25  22260.560000   
3  7372              Kota Parepare   q1-25 -20908.313313   
4  7601           Kabupaten Majene   q1-25 -56427.640000   

                                    kolom  \
0  adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
1  adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
2  adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
3  adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   
4  adhb_pi_q1-25_ril vs adhb_pi_q1-25_rev   

                                           evaluasi  
0  Nilai revisi lebih dari (+-30%) dari nilai rilis  
1  Nilai revisi lebih dari (+-30%) dari nilai rilis  
2  Nilai revisi lebih dari (+-30%) dari nilai rilis  
3  Nilai revisi lebih dari (+-30%) dari nilai rilis  
4  Nilai revisi lebih dari (+-30%) dari nilai rilis  


## Menuliskan hasil evaluasi kedalam file excel

In [74]:
write_to_excel(evaluasi, "kabupaten/kota")

Sheet 1803 ditambahkan ke data/output/evaluasi_18.xlsx
Sheet 7211 ditambahkan ke data/output/evaluasi_72.xlsx
Sheet 7271 ditambahkan ke data/output/evaluasi_72.xlsx
Sheet 7372 ditambahkan ke data/output/evaluasi_73.xlsx
Sheet 7601 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 7602 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 7603 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 7605 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 1221 ditambahkan ke data/output/evaluasi_12.xlsx
Sheet 1502 ditambahkan ke data/output/evaluasi_15.xlsx
Sheet 1808 ditambahkan ke data/output/evaluasi_18.xlsx
Sheet 3203 ditambahkan ke data/output/evaluasi_32.xlsx
Sheet 5319 ditambahkan ke data/output/evaluasi_53.xlsx
Sheet 7306 ditambahkan ke data/output/evaluasi_73.xlsx
Sheet 9201 ditambahkan ke data/output/evaluasi_92.xlsx
Sheet 3403 ditambahkan ke data/output/evaluasi_34.xlsx
Sheet 9102 ditambahkan ke data/output/evaluasi_91.xlsx
Sheet 9701 ditambahkan ke data/output/evaluasi_97.xlsx
Sheet 7317